In [ ]:
# ======================
# INSTALL
# ======================
!pip install librosa numpy pandas

# ======================
# IMPORTS
# ======================
import librosa
import numpy as np
import pandas as pd
from google.colab import files

# ======================
# UPLOAD AUDIO
# ======================
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

# ======================
# LOAD AUDIO
# ======================
y, sr = librosa.load(file_path, sr=22050)

# ======================
# FEATURE EXTRACTION
# ======================
hop = 512

rms = librosa.feature.rms(y=y, hop_length=hop).flatten()
centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop).flatten()
zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop).flatten()
rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop).flatten()
bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop).flatten()

S = np.abs(librosa.stft(y))
freqs = librosa.fft_frequencies(sr=sr)

bass = S[(freqs < 200)].mean(axis=0)
treble = S[(freqs >= 2000)].mean(axis=0)

# ======================
# NORMALIZATION
# ======================
def norm(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max()!=x.min() else np.zeros_like(x)

rms = norm(rms)
centroid = norm(centroid)
zcr = norm(zcr)
rolloff = norm(rolloff)
bandwidth = norm(bandwidth)
bass = norm(bass)
treble = norm(treble)

times = np.arange(len(rms)) * hop / sr

# ======================
# MOOD MAPPING FUNCTION
# ======================
def get_mood(r, c, z, b, tr, bw):

    mood = []

    # ======================
    # ENERGY (RMS)
    # ======================
    if r > 0.85:
        mood += ["explosive", "intense"]
    elif r > 0.65:
        mood += ["energetic", "driving"]
    elif r > 0.4:
        mood += ["moderate", "steady"]
    else:
        mood += ["calm", "ambient"]

    # ======================
    # PITCH / BRIGHTNESS (centroid)
    # ======================
    if c > 0.8:
        mood += ["sparkling", "brilliant"]
    elif c > 0.6:
        mood += ["bright", "clear"]
    elif c > 0.4:
        mood += ["balanced"]
    else:
        mood += ["dark", "warm"]

    # ======================
    # RHYTHM / TEXTURE (ZCR)
    # ======================
    if z > 0.75:
        mood += ["chaotic", "noisy"]
    elif z > 0.55:
        mood += ["textured", "grainy"]
    elif z > 0.3:
        mood += ["rhythmic"]
    else:
        mood += ["smooth", "fluid"]

    # ======================
    # BASS
    # ======================
    if b > 0.8:
        mood += ["bass-heavy", "thumping"]
    elif b > 0.6:
        mood += ["deep", "grounded"]
    elif b < 0.3:
        mood += ["lightweight", "thin"]

    # ======================
    # TREBLE
    # ======================
    if tr > 0.8:
        mood += ["piercing", "crisp"]
    elif tr > 0.6:
        mood += ["sharp", "articulate"]
    elif tr < 0.3:
        mood += ["soft", "muted"]

    # ======================
    # TIMBRE WIDTH (bandwidth)
    # ======================
    if bw > 0.8:
        mood += ["rich", "full"]
    elif bw > 0.6:
        mood += ["layered"]
    elif bw < 0.3:
        mood += ["minimal", "narrow"]

    # ======================
    # COMBINED HIGH-LEVEL EMOTION
    # ======================
    if r > 0.7 and c > 0.7:
        mood.append("uplifting")

    if r < 0.3 and c < 0.4:
        mood.append("melancholic")

    if b > 0.7 and r > 0.6:
        mood.append("powerful")

    if tr > 0.7 and z > 0.6:
        mood.append("aggressive")

    if r < 0.4 and bw < 0.4:
        mood.append("intimate")

    # ======================
    # REMOVE DUPES
    # ======================
    mood = list(set(mood))

    return ", ".join(mood)

# ======================
# BUILD TABLE
# ======================
data = []

for i in range(len(times)):
    mood = get_mood(
        rms[i],
        centroid[i],
        zcr[i],
        bass[i],
        treble[i],
        bandwidth[i]
    )

    data.append({
        "time_sec": round(times[i], 2),
        "energy": rms[i],
        "pitch": centroid[i],
        "rhythm": zcr[i],
        "bass": bass[i],
        "treble": treble[i],
        "timbre_width": bandwidth[i],
        "mood": mood
    })

df = pd.DataFrame(data)

# ======================
# OUTPUT
# ======================
print(df.head(20))

# Optional: save
df.to_csv("mood_table.csv", index=False)
files.download("mood_table.csv")

In [ ]:
#Emotions/Audio Combination
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import joblib

def generate_fused_mood_table(audio_path, model_path, scaler_path, output_name):
    # 1. SETUP
    model = tf.keras.models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    y, sr = librosa.load(audio_path, sr=22050)

    rows = []
    win_len, step = 10, 2 # 10s analysis window, sliding every 2s

    for start in range(0, int(librosa.get_duration(y=y, sr=sr)) - win_len, step):
        y_seg = y[int(start*sr) : int((start+win_len)*sr)]

        # 2. AI COMPONENT (High-Level Emotion)
        # Prepare Mel-spectrogram for your CNN
        S = librosa.feature.melspectrogram(y=y_seg, sr=sr, n_mels=128, hop_length=512)
        S_db = librosa.power_to_db(S, ref=np.max)
        input_data = np.resize(S_db, (1, 128, 431, 1))

        # Get Valence/Arousal (scaled 1-9)
        preds = scaler.inverse_transform(model.predict(input_data, verbose=0))[0]
        v, a = preds[0], preds[1]

        # 3. PHYSICAL COMPONENT (Low-Level Features)
        rms = np.mean(librosa.feature.rms(y=y_seg))
        bright = np.mean(librosa.feature.spectral_centroid(y=y_seg, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=y_seg))
        flat = np.mean(librosa.feature.spectral_flatness(y=y_seg))

        # 4. FUSED LABELING LOGIC (The "Mix")
        # We start with the AI quadrant and then "Sub-Label" based on features
        fused_mood = "Neutral"

        if a > 5.5: # HIGH AROUSAL BRANCH
            if v > 5.5:
                # High Energy + Positive = Happy.
                # If it's also very bright/loud, it's 'Majestic'
                fused_mood = "Majestic/Powerful" if (bright > 3000 or rms > 0.1) else "Cheerful/Uplifting"
            else:
                # High Energy + Negative = Tense.
                # If ZCR is high, it's 'Aggressive/Battle'. If ZCR is low, it's 'Anxious/Suspenseful'
                fused_mood = "Aggressive/Combat" if (zcr > 0.08 or flat > 0.02) else "Tense/Frantic"

        else: # LOW AROUSAL BRANCH
            if v > 5.5:
                # Low Energy + Positive = Calm.
                # If it's very 'pure' (flatness low), it's 'Ethereal'.
                fused_mood = "Ethereal/Dreamy" if flat < 0.005 else "Peaceful/Serene"
            else:
                # Low Energy + Negative = Sad.
                # If it's dark (low brightness), it's 'Somber'.
                fused_mood = "Dark/Gothic" if bright < 1200 else "Melancholic/Sad"

        rows.append({
            "time_sec": start,
            "mood": fused_mood,        # The NEW Fused Descriptor
            "v_ai": round(v, 2),       # Keeping the raw AI data for the visualizer to use
            "a_ai": round(a, 2),
            "energy": round(rms, 4),   # Keeping the raw Librosa data
            "brightness": round(bright, 1)
        })

    pd.DataFrame(rows).to_csv(output_name, index=False)
    print(f"Table built with Fused Mood descriptors: {output_name}")
